In [1]:
import logging
import pyreadstat
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
def leer_sav(ruta):
    df, meta = pyreadstat.read_sav(ruta)
    df.columns = df.columns.str.upper()
    return df, meta

In [3]:
# Configuración
KEY_HOGAR = ["CONGLOME", "VIVIENDA", "HOGAR"]
KEY_PERSONA = KEY_HOGAR + ["CODPERSO"]

V = dict(
    parentesco="P203", sexo="P207", edad="P208A",
    resid="P204", ausente="P205", presente="P206",      # definición de miembro del hogar
    lengua="P300A", nivel="P301A", grado="P301B",
    ocu="OCU500", cat_ocup="P507", horas="P513T", ingreso="I524A1",
    f_persona="FACTOR07", f_empleo="FAC500A",
    gasto="GASTO_ALIMENTOS",                              # = suma de la variable de gasto del 601
)

# P301A: 1 sin nivel, 2 inicial, 3 prim. incompleta, 4 prim. completa, 5 sec. incompleta,
# 6 sec. completa, 7 sup. no univ. incompleta, 8 sup. no univ. completa,
# 9 sup. univ. incompleta, 10 sup. univ. completa, 11 posgrado, 12 básica especial
NIVEL_CAT = {1: "sin nivel/inicial", 2: "sin nivel/inicial", 3: "primaria", 4: "primaria",
             5: "secundaria", 6: "secundaria", 7: "superior no univ.", 8: "superior no univ.",
             9: "superior univ.", 10: "superior univ.", 11: "posgrado"}
COMPLETO = {1: 0, 2: 0, 4: 6, 6: 11, 8: 14, 10: 16, 11: 18}      # años de estudio acumulados
INCOMPLETO = {3: (0, 5), 5: (6, 4), 7: (11, 3), 9: (11, 5)}      # nivel: (años base, máx. grados)


# Utilidades
def num(d, clave):
    return pd.to_numeric(d[V[clave]], errors="coerce")


def flag(cond, universo=None):
    """Indicador 0/1 que queda NA fuera de su universo (no lo convierte en 0)."""
    s = cond.astype("float64")
    if universo is not None:
        s = s.where(universo)
    return s.astype("Int8")


def validar_vars(df):
    faltan = sorted(c for c in set(V.values()) if c not in df.columns)
    if faltan:
        logging.warning("Variables de V que no existen en la base: %s", faltan)
    return faltan


def cuantiles_ponderados(x, w, qs):
    o = np.argsort(x.values)
    xs, c = x.values[o], np.cumsum(w.values[o]) / w.values.sum()
    return [xs[min(np.searchsorted(c, q), len(xs) - 1)] for q in qs]


def anios_estudio(nivel, grado):
    """Aproximación: años acumulados por nivel completo; en incompletos, base + grado aprobado."""
    out = nivel.map(COMPLETO).astype("float64")
    for n, (base, mx) in INCOMPLETO.items():
        m = nivel == n
        out.loc[m] = base + grado[m].clip(0, mx)
    return out

In [4]:
# Nivel persona
def features_persona(df):
    d = df.copy()
    edad, sexo, par = num(d, "edad"), num(d, "sexo"), num(d, "parentesco")
    resid, aus, pres = num(d, "resid"), num(d, "ausente"), num(d, "presente")

    # Miembro del hogar (definición INEI): residente habitual presente, o no residente presente >30 días
    d["es_miembro"] = (((resid == 1) & (aus == 2)) | ((resid == 2) & (pres == 1))).astype("int8")

    d["edad"] = edad.where(edad.between(0, 110))
    d["grupo_edad"] = pd.cut(d["edad"], [-1, 14, 29, 44, 59, 200],
                             labels=["0-14", "15-29", "30-44", "45-59", "60+"]).astype("string")
    d["mujer"] = flag(sexo == 2, sexo.notna())
    d["jefe_hogar"] = flag(par == 1, par.notna())

    # Educación (módulo 03)
    nivel, grado, lengua = num(d, "nivel"), num(d, "grado"), num(d, "lengua")
    d["anios_estudio"] = anios_estudio(nivel, grado)
    d["nivel_educ"] = nivel.map(NIVEL_CAT).astype("string")
    d["superior_alguna"] = flag(nivel.isin([7, 8, 9, 10, 11]), nivel.notna())
    d["superior_completa"] = flag(nivel.isin([8, 10, 11]), nivel.notna())
    d["lengua_nativa"] = flag(lengua.isin([1, 2, 3]), lengua.notna())

    # Empleo (módulo 05): solo definido para 14+; fuera del universo queda NA
    ocu, cat = num(d, "ocu"), num(d, "cat_ocup")
    d["pea"] = flag(ocu.isin([1, 2, 3]), ocu.notna())
    d["ocupado"] = flag(ocu == 1, ocu.notna())
    d["desocupado"] = flag(ocu.isin([2, 3]), ocu.notna())
    d["asalariado"] = flag(cat.isin([3, 4]), cat.notna() & (ocu == 1))
    d["independiente"] = flag(cat == 2, cat.notna() & (ocu == 1))
    d["horas_sem"] = num(d, "horas").where(lambda s: s.between(1, 168))
    ing = num(d, "ingreso").where(lambda s: s > 0)
    d["ingreso_laboral"] = ing
    d["log_ingreso"] = np.log(ing)
    d["ingreso_hora"] = ing / (d["horas_sem"] * 4.33)       # válido solo si el ingreso es mensual
    d["exp_pot"] = (d["edad"] - d["anios_estudio"] - 6).clip(lower=0)
    return d


# Nivel hogar
def features_hogar(d):
    """Una fila por hogar, calculada solo con miembros del hogar."""
    m = d[d["es_miembro"] == 1].copy()
    m["_men"] = m["edad"] < 15
    m["_may"] = m["edad"] >= 65
    m["_act"] = m["edad"].between(15, 64)
    m["_ocu"] = m["ocupado"] == 1
    g = m.groupby(KEY_HOGAR)

    h = g.agg(tam_hogar=("CODPERSO", "size"), n_menores=("_men", "sum"), n_mayores65=("_may", "sum"),
              n_15_64=("_act", "sum"), n_ocupados=("_ocu", "sum"), educ_max=("anios_estudio", "max"))
    h["ingreso_laboral_hogar"] = g["ingreso_laboral"].sum(min_count=1)

    hogar_cols = [c for c in ("UBIGEO", "DOMINIO", "ESTRATO", V["f_persona"], V["gasto"]) if c in m.columns]
    h = h.join(g[hogar_cols].first())

    jefe = m[m["jefe_hogar"] == 1].drop_duplicates(KEY_HOGAR).set_index(KEY_HOGAR)
    h = h.join(jefe[["edad", "mujer", "anios_estudio", "ocupado"]].rename(columns={
        "edad": "edad_jefe", "mujer": "jefe_mujer",
        "anios_estudio": "anios_estudio_jefe", "ocupado": "jefe_ocupado"}))

    h["factor_hogar"] = h[V["f_persona"]]        # CONFIRMAR: factor de hogar correcto para el 02
    h["ratio_dependencia"] = (h["n_menores"] + h["n_mayores65"]) / h["n_15_64"].where(h["n_15_64"] > 0)
    h["carga_economica"] = h["tam_hogar"] / h["n_ocupados"].where(h["n_ocupados"] > 0)
    h["gasto_pc"] = h[V["gasto"]] / h["tam_hogar"]
    h["log_gasto_pc"] = np.log1p(h["gasto_pc"])
    h["ingreso_pc"] = h["ingreso_laboral_hogar"] / h["tam_hogar"]
    h["unipersonal"] = (h["tam_hogar"] == 1).astype("int8")
    h["con_menores"] = (h["n_menores"] > 0).astype("int8")
    h["con_mayores65"] = (h["n_mayores65"] > 0).astype("int8")

    # Quintiles de gasto en alimentos per cápita, ponderados (proxy, no es el gasto total)
    ok = h["gasto_pc"].notna() & h["factor_hogar"].notna()
    cortes = cuantiles_ponderados(h.loc[ok, "gasto_pc"], h.loc[ok, "factor_hogar"], [.2, .4, .6, .8])
    q = pd.Series(np.searchsorted(cortes, h["gasto_pc"].fillna(0)) + 1, index=h.index)
    h["quintil_gasto"] = q.where(h["gasto_pc"].notna()).astype("Int8")

    e, men = h["edad_jefe"], h["n_menores"]
    h["ciclo_vida"] = np.select(
        [e.isna(), (e < 30) & (men == 0), (men > 0) & (e < 45), men > 0, e >= 60],
        ["sin dato", "joven sin menores", "familia joven", "familia consolidada", "adulto mayor sin menores"],
        default="adulto sin menores")
    return h.reset_index()


def unir_hogar_a_persona(d, h, cols=("tam_hogar", "quintil_gasto", "ciclo_vida", "ratio_dependencia")):
    return d.merge(h[KEY_HOGAR + list(cols)], on=KEY_HOGAR, how="left", validate="many_to_one")


# Catálogo de derivadas
DERIVADAS = {
    # nombre: (etiqueta, nivel, universo)
    "es_miembro": ("Miembro del hogar (definición INEI)", "persona", "todos"),
    "edad": ("Edad en años (0-110)", "persona", "todos"),
    "grupo_edad": ("Grupo de edad", "persona", "todos"),
    "mujer": ("1 = mujer", "persona", "todos"),
    "jefe_hogar": ("1 = jefe(a) de hogar", "persona", "todos"),
    "anios_estudio": ("Años de estudio (aprox. por nivel y grado)", "persona", "3+ años"),
    "nivel_educ": ("Nivel educativo agrupado", "persona", "3+ años"),
    "superior_alguna": ("1 = algún estudio superior", "persona", "3+ años"),
    "superior_completa": ("1 = superior completa o posgrado", "persona", "3+ años"),
    "lengua_nativa": ("1 = lengua materna nativa", "persona", "3+ años"),
    "pea": ("1 = población económicamente activa", "persona", "14+ años"),
    "ocupado": ("1 = ocupado", "persona", "14+ años"),
    "desocupado": ("1 = desocupado (abierto u oculto)", "persona", "14+ años"),
    "asalariado": ("1 = empleado u obrero", "persona", "ocupados"),
    "independiente": ("1 = trabajador independiente", "persona", "ocupados"),
    "horas_sem": ("Horas semanales en ocupación principal (1-168)", "persona", "ocupados"),
    "ingreso_laboral": ("Ingreso por trabajo principal (>0)", "persona", "ocupados"),
    "log_ingreso": ("Logaritmo del ingreso laboral", "persona", "ocupados"),
    "ingreso_hora": ("Ingreso laboral por hora (no usar como predictor del ingreso)", "persona", "ocupados"),
    "exp_pot": ("Experiencia potencial: edad - años de estudio - 6", "persona", "todos"),
    "tam_hogar": ("Número de miembros del hogar", "hogar", "hogares"),
    "n_menores": ("Miembros menores de 15 años", "hogar", "hogares"),
    "n_mayores65": ("Miembros de 65 años o más", "hogar", "hogares"),
    "n_15_64": ("Miembros de 15 a 64 años", "hogar", "hogares"),
    "n_ocupados": ("Miembros ocupados", "hogar", "hogares"),
    "educ_max": ("Máximo de años de estudio en el hogar", "hogar", "hogares"),
    "ingreso_laboral_hogar": ("Suma de ingresos laborales del hogar", "hogar", "hogares"),
    "edad_jefe": ("Edad del jefe de hogar", "hogar", "hogares"),
    "jefe_mujer": ("1 = jefe de hogar mujer", "hogar", "hogares"),
    "anios_estudio_jefe": ("Años de estudio del jefe", "hogar", "hogares"),
    "jefe_ocupado": ("1 = jefe ocupado", "hogar", "hogares"),
    "factor_hogar": ("Factor de expansión usado para hogares", "hogar", "hogares"),
    "ratio_dependencia": ("(menores 15 + mayores 65) / población 15-64", "hogar", "hogares"),
    "carga_economica": ("Miembros por ocupado", "hogar", "hogares con ocupados"),
    "gasto_pc": ("Gasto en alimentos per cápita (solo módulo 07)", "hogar", "hogares"),
    "log_gasto_pc": ("log(1 + gasto per cápita)", "hogar", "hogares"),
    "ingreso_pc": ("Ingreso laboral per cápita del hogar", "hogar", "hogares"),
    "unipersonal": ("1 = hogar de una persona", "hogar", "hogares"),
    "con_menores": ("1 = hogar con menores de 15", "hogar", "hogares"),
    "con_mayores65": ("1 = hogar con mayores de 65", "hogar", "hogares"),
    "quintil_gasto": ("Quintil ponderado de gasto en alimentos per cápita", "hogar", "hogares"),
    "ciclo_vida": ("Ciclo de vida del hogar (edad del jefe y menores)", "hogar", "hogares"),
}
LABELS_DERIVADAS = {k: v[0] for k, v in DERIVADAS.items()}
FLAGS = ["mujer", "jefe_hogar", "superior_alguna", "superior_completa", "lengua_nativa", "pea",
         "ocupado", "desocupado", "asalariado", "independiente", "es_miembro", "jefe_mujer",
         "jefe_ocupado", "unipersonal", "con_menores", "con_mayores65"]
VL_FLAGS = {c: {0: "No", 1: "Sí"} for c in FLAGS}


In [5]:

RUTA = "/Users/haeidyvilcapazahuaman/Documents/GitHub/Maestria_proyecto/data/processed/enaho2024_mod2_3_5_7.sav"

base_sav, meta_sav = leer_sav(RUTA)
validar_vars(base_sav)                       # lista de variables de que no existen

[]

In [7]:
def exportar_sav(df, ruta, labels, vlabels):
    d = df.copy()
    for c in d.columns:
        s = d[c]
        if pd.api.types.is_bool_dtype(s) or (
                pd.api.types.is_extension_array_dtype(s) and pd.api.types.is_numeric_dtype(s)):
            d[c] = s.astype("float64")                       # Int8/boolean con NA -> float
        elif isinstance(s.dtype, pd.CategoricalDtype) or pd.api.types.is_string_dtype(s):
            d[c] = s.astype("object").where(s.notna(), "").astype(str)
    pyreadstat.write_sav(
        d, ruta,
        column_labels=[(labels.get(c) or "")[:256] for c in d.columns],
        variable_value_labels={k: v for k, v in vlabels.items() if k in d.columns},
        compress=True,
    )
    logging.info("Exportado %s: %d filas x %d columnas", ruta, *d.shape)


In [8]:
personas = features_persona(base_sav)
hogares = features_hogar(personas)
personas = unir_hogar_a_persona(personas, hogares)


In [9]:
def etiquetas_originales(meta, columnas):
    cols = set(columnas)
    labels, vlabels = {}, {}
    for v in meta.column_names:
        c = v.upper()
        if c in cols:
            labels[c] = meta.column_names_to_labels.get(v) or ""
            if meta.variable_value_labels.get(v):
                vlabels[c] = meta.variable_value_labels[v]
    return labels, vlabels

In [ ]:

# Guardar con etiquetas de las derivadas
labels, vlabels = etiquetas_originales(meta_sav, base_sav.columns)
labels.update({k: v[0] for k, v in DERIVADAS.items()})
vlabels.update(VL_FLAGS)
exportar_sav(personas, f"/Users/haeidyvilcapazahuaman/Documents/GitHub/Maestria_proyecto/data/processed//enaho2024_personas_fe.sav", labels, vlabels)
exportar_sav(hogares, f"/Users/haeidyvilcapazahuaman/Documents/GitHub/Maestria_proyecto/data/processed//enaho2024_hogares_fe.sav", labels, vlabels)